# Быстрый старт с HTTP API rulemma.ru

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DimaKorshikov/rulemma/blob/main/examples/notebooks/01_api_quickstart.ipynb)

Знакомство с облачным API `rulemma`: проверка доступности и тарифов, вызов без регистрации через песочницу, авторизация Bearer-токеном, пакетная лемматизация и ремонт RU/EN омоглифов. Работает в Google Colab и локальном Jupyter без установки Go и сторонних пакетов.

### Регистрация и доступ к HTTP API

- Личный кабинет и бесплатный API-ключ: **[rulemma.ru/account/](https://rulemma.ru/account/)** (вход по email, Яндекс ID или Google)
- Документация HTTP API: [rulemma.ru/api/](https://rulemma.ru/api/) · Тарифы и лимиты: [rulemma.ru/pricing/](https://rulemma.ru/pricing/) · OpenAPI: [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Базовый адрес API: `https://rulemma.ru/v1` · Авторизация: заголовок `Authorization: Bearer rlm_live_...` (передавать ключ в URL запрещено)

Как подключить ключ:
1. Создайте бесплатный аккаунт на [rulemma.ru/account/](https://rulemma.ru/account/) и скопируйте токен `rlm_live_...`.
2. Сохраните его в переменную окружения `RULEMMA_API_KEY` (в Google Colab удобно положить токен в панель `Secrets` под именем `RULEMMA_API_KEY` или ввести в ячейке настройки).
3. Без ключа ячейки песочницы и общие примеры сработают автоматически (до 10 строк за вызов); с токеном включаются полные батч-эндпоинты.

### Содержание тетрадки

1. Настройка подключения и проверка статуса API (`/v1/health`, `/v1/tariffs`)
2. Пробный вызов без ключа через публичную песочницу (`/v1/sandbox/test`)
3. Проверка версии словаря и остатка квоты (`/v1/info`, `/v1/usage`)
4. Пакетная лемматизация и смысловые ключи (`/v1/lemmatize`)
5. Ремонт смешанных кириллица+латиница омоглифов (`/v1/homoglyphs`)

## 1. Настройка клиента (только стандартная библиотека Python)

При желании токен можно задать до запуска тетрадки (`export RULEMMA_API_KEY=rlm_live_...`) или положить в Google Colab → Secrets.

In [ ]:
import json
import os
import urllib.error
import urllib.request

BASE_URL = os.environ.get("RULEMMA_API_BASE", "https://rulemma.ru/v1").rstrip("/")
API_KEY = os.environ.get("RULEMMA_API_KEY", "").strip()

# В Google Colab ключ можно автоматически прочитать из вкладки Secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        API_KEY = (userdata.get("RULEMMA_API_KEY") or "").strip()
    except Exception:
        pass

HAS_KEY = bool(API_KEY)
print("Базовый URL:", BASE_URL)
print(
    "Режим:",
    "боевой API с Bearer-токеном" if HAS_KEY else "публичная песочница без ключа (до 10 строк)",
)
if not HAS_KEY:
    print("Получить бесплатный API-ключ: https://rulemma.ru/account/")


def api_request(
    method: str,
    path: str,
    payload: dict | None = None,
    *,
    auth: bool = True,
    timeout: int = 20,
) -> tuple[dict, dict[str, str]]:
    """Выполняет HTTP-запрос к rulemma API без сторонних зависимостей."""
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "rulemma-jupyter-example/1.0",
    }
    if auth:
        if not API_KEY:
            raise RuntimeError(
                "Требуется RULEMMA_API_KEY. Выпустите бесплатный токен на https://rulemma.ru/account/"
            )
        headers["Authorization"] = f"Bearer {API_KEY}"

    body = None
    if payload is not None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json; charset=utf-8"

    import time as _time

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    for attempt in range(2):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                resp_headers = {k.lower(): v for k, v in resp.headers.items()}
                return json.loads(resp.read().decode("utf-8")), resp_headers
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
            if exc.code == 429 and attempt == 0:
                wait = min(12, max(1, int(exc.headers.get("Retry-After", "4"))))
                print(f"Лимит скорости (HTTP 429), пауза {wait} с...")
                _time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {exc.code} {path}: {raw}") from exc
    raise RuntimeError(f"Не удалось выполнить {method} {path}")

## 2. Проверка доступности API и актуальных тарифов (без ключа)

Эндпоинты `GET /v1/health` и `GET /v1/tariffs` открыты без авторизации: удобно проверять живость сервиса и актуальные лимиты планов.

In [ ]:
health, _ = api_request("GET", "/health", auth=False)
tariffs, _ = api_request("GET", "/tariffs", auth=False)

print("Статус API:", health)
print("Единица тарификации:", tariffs.get("unit_definition", "1 строка до 256 символов"))
print("\nДоступные планы:")
for plan in tariffs.get("plans", []):
    name = plan.get("name", plan.get("id"))
    units = plan.get("monthly_units", 0)
    batch = plan.get("max_batch", 0)
    rub = plan.get("monthly_rub", 0)
    price = "по запросу" if plan.get("custom") or rub is None else ("0 ₽/мес" if rub == 0 else f"{rub} ₽/мес")
    print(f"  - {name:12s} | квота: {units:>10,} | батч: до {batch:>6,} | {price}")

## 3. Первый тест без регистрации (`POST /v1/sandbox/test`)

Публичная песочница принимает до 10 фраз за вызов и не требует токена — идеально, чтобы проверить качество на своих запросах перед регистрацией.

In [ ]:
sample_queries = [
    "Купить зимние кроссовки найк",
    "Кроссовки на зиму найк",
    "Спрей от комаров 100 мл",
    "Шина 35 см на бензопилу",
    "Оцинкованные ведра для дачи",
]

sandbox_resp, _ = api_request(
    "POST",
    "/sandbox/test",
    {
        "method": "lemmatize",
        "queries": sample_queries,
        "profile": "technical",
    },
    auth=False,
)

print(
    f"Версия на сервере: rulemma {sandbox_resp.get('rulemma_version')} "
    f"(словарь {sandbox_resp.get('dict_version')})"
)
for row in sandbox_resp["items"]:
    print(f"  {row['input']}")
    print(f"    lemma:        {row['lemmatized']}")
    print(f"    semantic_key: {row['semantic_key']}")

## 4. Информация о словаре и остатке квоты (`GET /v1/info`, `GET /v1/usage`)

После подключения бесплатного ключа с [rulemma.ru/account/](https://rulemma.ru/account/) доступны счётчики словаря, текущий тариф, лимит батча и остаток единиц.

In [ ]:
if HAS_KEY:
    info, _ = api_request("GET", "/info")
    usage, _ = api_request("GET", "/usage")
    q = usage["quota"]
    print(f"Движок: v{info['version']} (ABI {info['abi_version']}), словарь {info['dict_version']}")
    print("Таблицы словаря:", info["counts"])
    print(
        f"Тариф: {usage['plan_name']} | батч до {usage['max_batch']:,} | "
        f"RPS: {usage['rate_limit_rps']} | остаток: {q['remaining']:,} из {q['limit']:,}"
    )
else:
    print("Пропущено: задайте RULEMMA_API_KEY (бесплатная регистрация: https://rulemma.ru/account/).")

## 5. Боевой вызов `/v1/lemmatize` и ремонт омоглифов `/v1/homoglyphs`

Если `RULEMMA_API_KEY` задан, вызываются рабочие эндпоинты и выводятся заголовки учёта квоты (`X-Units-Charged`, `X-Quota-Remaining`). Без ключа запрос автоматически идёт в песочницу.

In [ ]:
dirty_texts = [
    "Кабель 5cм intеl рrо",
    "Ноутбук аsus ryzеn 7",
    "Наушники sоny 3,5 мм",
]

if HAS_KEY:
    homo_resp, hdrs = api_request("POST", "/homoglyphs", {"texts": dirty_texts})
    fixed_texts = homo_resp["items"]
    print(
        f"/v1/homoglyphs: списано {hdrs.get('x-units-charged', '?')} ед., "
        f"остаток {hdrs.get('x-quota-remaining', '?')}"
    )
else:
    homo_resp, _ = api_request(
        "POST",
        "/sandbox/test",
        {"method": "homoglyphs", "queries": dirty_texts},
        auth=False,
    )
    fixed_texts = [row["result"] for row in homo_resp["items"]]
    print("Режим песочницы (/v1/sandbox/test -> homoglyphs):")

for before, after in zip(dirty_texts, fixed_texts):
    print(f"  {before!r}  ->  {after!r}")

---

## Что дальше

- Следующая тетрадка: [02_query_dedup_and_technical_keys.ipynb](02_query_dedup_and_technical_keys.ipynb)
- Все Jupyter-тетрадки с HTTP API: [examples/notebooks/README.md](README.md)
- Регистрация и управление API-ключами: [rulemma.ru/account/](https://rulemma.ru/account/)
- Документация эндпоинтов и форматов ошибок: [rulemma.ru/api/](https://rulemma.ru/api/) · [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Нужны миллионы строк локально без сети (Go / Python CGO / CLI)? Смотрите локальные примеры [examples/01..13](../README.md).